[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-2-CORRECTED_executed.ipynb)

---


# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 2: List Scheduling Algorithm

### Goal and scope

This notebook implements a list-scheduling heuristic for rearranging containers among stacks. A container may be moved only when it is at the top of its stack, and a destination stack must have an available tier. At each iteration, the algorithm scores the feasible moves, selects the move with the highest score, and updates the configuration. It stops when there are no priority violations, no feasible moves, or the move limit is reached.

The method is a greedy heuristic: it makes one locally selected move at a time. It does not solve a global optimization model and does not guarantee the fewest moves or a feasible final arrangement. The score combines immediate change in priority violations with stack balance, a movement-efficiency term, and a priority bonus.

### Background: stacks, tiers, and priorities

A **stack** is a vertical column of containers. Tiers are numbered from the bottom, starting at tier 0. Each container has an integer priority; a smaller number means that the container has higher retrieval priority. In a stack, a lower-priority container should not block a higher-priority one. The example uses compact stacks: there are no empty tiers below an occupied tier.

A **priority violation** is an adjacent pair in one stack for which the lower container has a numerically larger priority than the container directly above it. With compact stacks, counting these adjacent inversions measures the local blocking conflicts in the configuration. Moving only a top container preserves accessibility: no container below it has to be lifted first.

### Scoring and move selection

Let \(x\) denote the current bay configuration and \(x'\) the configuration after a candidate move. Let \(V(x)\) count priority violations, and let \(h_s(x)\) be the height of stack \(s\). The average stack height is

$$
\bar{h}(x) = \frac{1}{S}\sum_{s=0}^{S-1} h_s(x),
$$

where \(S\) is the number of stacks. For a move from stack \(a\) to stack \(b\), the balance measure before the move is

$$
B(x;a,b) = |h_a(x)-\bar{h}(x)| + |h_b(x)-\bar{h}(x)|.
$$

The move score implemented below is

$$
\begin{aligned}
\operatorname{score}(x,m) ={}& 50\bigl(V(x)-V(x')\bigr) \\
&+20\bigl(B(x;a,b)-B(x';a,b)\bigr) \\
&+\max\bigl(0,\,15-2|a-b|-t_a\bigr) \\
&+15\frac{10-p_m}{10}.
\end{aligned}
$$

Here \(m\) is the candidate move, \(t_a\) is its source tier, and \(p_m\) is the moved container's priority. The first term rewards an actual reduction in the total violation count; it can be negative if a move creates violations. The second term rewards a reduction in the combined deviation of the source and destination heights from the average. The third term is a nonnegative efficiency score based on the stack-index distance and source tier. The final term is a priority bonus, using the example's priority scale of 1 to 10. The weights are heuristic scoring weights, not probabilities or units of physical cost.

At each iteration, the algorithm forms a list of feasible candidate moves and selects one with the maximum score. If scores tie, Python's `max` selects the first candidate in the generated order. This local choice does not imply that the total score, number of future moves, or final solution quality is globally optimal.

### Algorithm and computational effort

The algorithm starts with the supplied configuration. Its base condition is a configuration with zero priority violations. Otherwise, it generates feasible moves by examining each stack's top container and each other stack with free capacity, scores those candidates, and executes the highest-scoring move. The move limit bounds the number of iterations and prevents an unbounded sequence of heuristic moves.

For \(n\) containers, \(S\) stacks, \(T\) tiers, and at most \(M\) moves, this direct implementation takes at most \(O\bigl(MS^2(Sn+ST)\bigr)\) time under its repeated height and violation-count calculations. This is a bound for the implementation's candidate generation and scoring, not a guarantee of solution quality. The configuration requires \(O(n+ST)\) space, excluding stored move and log histories.

### Modeling assumptions

- All stacks have the same tier capacity, and each container occupies one tier.
- Containers are stored in compact stacks, with occupied tiers numbered consecutively from the bottom.
- Only the top container of a stack can be moved, and a move cannot return a container to its source stack as part of the same operation.
- Stack-index distance is used only as a simple movement-efficiency proxy; actual travel paths, handling times, equipment conflicts, and operational costs are not modeled.
- Priorities are integer ranks, with smaller values indicating higher retrieval priority. The priority bonus is scaled for values from 1 to 10.

### Concrete example

The example has three stacks and five containers:

- Stack 0: bottom \(B(2)\), top \(E(5)\).
- Stack 1: bottom \(D(4)\), top \(A(1)\).
- Stack 2: \(C(3)\) at the bottom.

The number in parentheses is the container priority. The code constructs this configuration, reports its violations, runs the list scheduler, and displays the resulting moves and bay configurations.

In [ ]:
import time
from dataclasses import dataclass, replace
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """A container's identity, retrieval priority, and current position."""

    id: str
    priority: int  # Smaller values indicate higher retrieval priority.
    current_stack: int
    current_tier: int


@dataclass
class Move:
    """A proposed move from one stack position to another."""

    container_id: str
    from_stack: int
    from_tier: int
    to_stack: int
    to_tier: int
    score: float


class ListScheduler:
    """Greedy list scheduler for the yard pre-marshalling problem."""

    def __init__(self, stacks: int, tiers: int):
        if stacks <= 0 or tiers <= 0:
            raise ValueError("The numbers of stacks and tiers must be positive.")

        self.stacks = stacks
        self.tiers = tiers
        self.move_history: List[Move] = []
        self.execution_log: List[str] = []

    def validate_configuration(self, containers: List[Container]) -> None:
        """Check that positions describe feasible, compact stacks."""
        identifiers = [container.id for container in containers]
        if len(identifiers) != len(set(identifiers)):
            raise ValueError("Container identifiers must be unique.")

        occupied_positions = set()
        stack_tiers = [[] for _ in range(self.stacks)]

        for container in containers:
            if not 0 <= container.current_stack < self.stacks:
                raise ValueError(f"Container {container.id} has an invalid stack.")
            if not 0 <= container.current_tier < self.tiers:
                raise ValueError(f"Container {container.id} has an invalid tier.")
            if container.priority <= 0:
                raise ValueError(f"Container {container.id} must have a positive priority.")

            position = (container.current_stack, container.current_tier)
            if position in occupied_positions:
                raise ValueError("Two containers cannot occupy the same position.")

            occupied_positions.add(position)
            stack_tiers[container.current_stack].append(container.current_tier)

        for tiers_in_stack in stack_tiers:
            ordered_tiers = sorted(tiers_in_stack)
            if ordered_tiers != list(range(len(ordered_tiers))):
                raise ValueError("Each stack must be compact, with no gaps between tiers.")

    def get_stack_height(self, containers: List[Container], stack: int) -> int:
        """Return the number of containers currently in a stack."""
        return sum(container.current_stack == stack for container in containers)

    def get_top_container(
        self, containers: List[Container], stack: int
    ) -> Optional[Container]:
        """Return the highest-tier container, or None for an empty stack."""
        stack_containers = [
            container
            for container in containers
            if container.current_stack == stack
        ]
        if not stack_containers:
            return None
        return max(stack_containers, key=lambda container: container.current_tier)

    def count_priority_violations(self, containers: List[Container]) -> int:
        """Count adjacent priority inversions in the current stacks."""
        grid = [[None for _ in range(self.tiers)] for _ in range(self.stacks)]
        for container in containers:
            grid[container.current_stack][container.current_tier] = container.priority

        violations = 0
        for stack in range(self.stacks):
            for tier in range(self.tiers - 1):
                lower_priority = grid[stack][tier]
                upper_priority = grid[stack][tier + 1]

                if (
                    lower_priority is not None
                    and upper_priority is not None
                    and lower_priority > upper_priority
                ):
                    violations += 1

        return violations

    def calculate_move_score(self, containers: List[Container], move: Move) -> float:
        """Score a feasible move using violation, balance, efficiency, and priority terms."""
        container = next(
            item for item in containers if item.id == move.container_id
        )

        # The score uses the actual resulting configuration so that its violation term
        # rewards reductions and penalizes increases in the counted conflicts.
        violations_before = self.count_priority_violations(containers)
        moved_containers = self.execute_move(containers, move)
        violations_after = self.count_priority_violations(moved_containers)
        violation_reduction = violations_before - violations_after
        score = 50 * violation_reduction

        from_height = self.get_stack_height(containers, move.from_stack)
        to_height = self.get_stack_height(containers, move.to_stack)
        average_height = sum(
            self.get_stack_height(containers, stack)
            for stack in range(self.stacks)
        ) / self.stacks

        balance_before = (
            abs(from_height - average_height)
            + abs(to_height - average_height)
        )
        balance_after = (
            abs(from_height - 1 - average_height)
            + abs(to_height + 1 - average_height)
        )
        score += 20 * (balance_before - balance_after)

        distance = abs(move.from_stack - move.to_stack)
        efficiency = 15 - (2 * distance) - move.from_tier
        score += max(0, efficiency)

        priority_bonus = 15 * (10 - container.priority) / 10
        score += priority_bonus

        return score

    def generate_valid_moves(self, containers: List[Container]) -> List[Move]:
        """Generate and score moves of accessible top containers."""
        moves = []

        for stack in range(self.stacks):
            top_container = self.get_top_container(containers, stack)
            if top_container is None:
                continue

            for target_stack in range(self.stacks):
                if target_stack == stack:
                    continue

                target_height = self.get_stack_height(containers, target_stack)
                if target_height >= self.tiers:
                    continue

                move = Move(
                    container_id=top_container.id,
                    from_stack=stack,
                    from_tier=top_container.current_tier,
                    to_stack=target_stack,
                    to_tier=target_height,
                    score=0.0,
                )
                move.score = self.calculate_move_score(containers, move)
                moves.append(move)

        return moves

    def execute_move(self, containers: List[Container], move: Move) -> List[Container]:
        """Return a new container list with the selected move applied."""
        new_containers = [replace(container) for container in containers]
        moved_container = next(
            (container for container in new_containers if container.id == move.container_id),
            None,
        )
        if moved_container is None:
            raise ValueError(f"Unknown container identifier: {move.container_id}")

        moved_container.current_stack = move.to_stack
        moved_container.current_tier = move.to_tier
        return new_containers

    def solve(
        self, initial_containers: List[Container], max_moves: int = 100
    ) -> Tuple[List[Container], List[Move]]:
        """Run greedy list scheduling until a stopping condition is reached."""
        if max_moves < 0:
            raise ValueError("max_moves must be nonnegative.")

        containers = [replace(container) for container in initial_containers]
        self.validate_configuration(containers)
        self.move_history = []
        self.execution_log = []

        iteration = 0
        while iteration < max_moves:
            violations = self.count_priority_violations(containers)
            if violations == 0:
                self.execution_log.append(
                    f"Iteration {iteration}: No priority violations remain."
                )
                break

            valid_moves = self.generate_valid_moves(containers)
            if not valid_moves:
                self.execution_log.append(
                    f"Iteration {iteration}: No feasible moves are available."
                )
                break

            # Candidate generation order resolves equal scores deterministically.
            best_move = max(valid_moves, key=lambda candidate: candidate.score)
            containers = self.execute_move(containers, best_move)
            self.move_history.append(best_move)

            new_violations = self.count_priority_violations(containers)
            self.execution_log.append(
                f"Iteration {iteration}: Move {best_move.container_id} from "
                f"stack {best_move.from_stack} to stack {best_move.to_stack}, "
                f"score={best_move.score:.1f}, violations: "
                f"{violations} -> {new_violations}"
            )
            iteration += 1

        if (
            iteration >= max_moves
            and self.count_priority_violations(containers) > 0
        ):
            self.execution_log.append(
                f"Move limit reached after {max_moves} moves."
            )

        return containers, self.move_history

In [ ]:
def visualize_bay_state(
    containers: List[Container],
    stacks: int,
    tiers: int,
    title: str,
    ax=None,
) -> None:
    """Draw a bay configuration and identify priorities by color and label."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container

    priorities = sorted({container.priority for container in containers})
    if priorities:
        minimum_priority = min(priorities)
        maximum_priority = max(priorities)
        priority_range = max(1, maximum_priority - minimum_priority)
    else:
        minimum_priority = 0
        priority_range = 1

    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            position = (stack * 1.2, tier * 0.8)

            if container is not None:
                color_intensity = (
                    container.priority - minimum_priority
                ) / priority_range
                color = plt.cm.RdYlBu_r(color_intensity)
                rectangle = plt.Rectangle(
                    position,
                    1.0,
                    0.6,
                    facecolor=color,
                    edgecolor="black",
                    linewidth=2,
                )
                ax.add_patch(rectangle)
                ax.text(
                    stack * 1.2 + 0.5,
                    tier * 0.8 + 0.3,
                    f"{container.id}\n({container.priority})",
                    ha="center",
                    va="center",
                    fontweight="bold",
                    fontsize=10,
                )
            else:
                rectangle = plt.Rectangle(
                    position,
                    1.0,
                    0.6,
                    facecolor="lightgray",
                    edgecolor="gray",
                    linewidth=1,
                    alpha=0.3,
                )
                ax.add_patch(rectangle)

    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xlabel("Stack index")
    ax.set_ylabel("Tier index (bottom = 0)")
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    for stack in range(stacks):
        ax.text(stack * 1.2 + 0.5, -0.3, f"S{stack}", ha="center", fontweight="bold")
    for tier in range(tiers):
        ax.text(-0.3, tier * 0.8 + 0.3, f"T{tier}", ha="center", fontweight="bold")

    if priorities:
        legend_handles = []
        for priority in priorities:
            color_intensity = (priority - minimum_priority) / priority_range
            legend_handles.append(
                Patch(
                    facecolor=plt.cm.RdYlBu_r(color_intensity),
                    edgecolor="black",
                    label=str(priority),
                )
            )
        ax.legend(
            handles=legend_handles,
            title="Priority (smaller = higher)",
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
        )


def visualize_algorithm_progression(
    initial_containers: List[Container],
    final_containers: List[Container],
    move_history: List[Move],
    stacks: int,
    tiers: int,
) -> None:
    """Show the initial state and selected states from the move sequence."""
    n_moves = len(move_history)
    if n_moves == 0:
        return

    n_stages = min(5, n_moves + 1)
    figure, axes = plt.subplots(
        1, n_stages, figsize=(5 * n_stages, 6), squeeze=False
    )
    axes = axes[0]

    visualize_bay_state(
        initial_containers, stacks, tiers, "Initial state", axes[0]
    )

    for stage in range(1, n_stages):
        moves_to_apply = round(stage * n_moves / (n_stages - 1))
        stage_containers = [replace(container) for container in initial_containers]

        for move in move_history[:moves_to_apply]:
            moved_container = next(
                container
                for container in stage_containers
                if container.id == move.container_id
            )
            moved_container.current_stack = move.to_stack
            moved_container.current_tier = move.to_tier

        if stage == n_stages - 1:
            stage_containers = [replace(container) for container in final_containers]
            title = f"Final state ({n_moves} moves)"
        else:
            title = f"After {moves_to_apply} moves"

        visualize_bay_state(
            stage_containers, stacks, tiers, title, axes[stage]
        )

    figure.tight_layout()
    plt.show()

In [ ]:
print("=" * 60)
print("YARD PRE-MARSHALLING: LIST SCHEDULING")
print("=" * 60)

containers = [
    Container("E", 5, 0, 1),
    Container("B", 2, 0, 0),
    Container("A", 1, 1, 1),
    Container("D", 4, 1, 0),
    Container("C", 3, 2, 0),
]

stacks, tiers = 3, 4
scheduler = ListScheduler(stacks, tiers)
scheduler.validate_configuration(containers)

print("\nProblem configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers per stack: {tiers}")
print(f"- Containers: {len(containers)}")

print("\nInitial container positions:")
for container in containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

initial_violations = scheduler.count_priority_violations(containers)
print(f"\nInitial priority violations: {initial_violations}")

visualize_bay_state(
    containers, stacks, tiers, "Initial bay configuration"
)

In [ ]:
scheduler = ListScheduler(stacks, tiers)

print("\n" + "=" * 60)
print("LIST SCHEDULING EXECUTION")
print("=" * 60)

start_time = time.time()
final_containers, move_history = scheduler.solve(containers, max_moves=50)
execution_time = time.time() - start_time

print(f"\nElapsed execution time: {execution_time:.4f} seconds")
print(f"Moves executed: {len(move_history)}")

print("\nExecution log:")
for log_entry in scheduler.execution_log:
    print(f"  {log_entry}")

In [ ]:
print("\n" + "=" * 60)
print("FINAL CONFIGURATION ANALYSIS")
print("=" * 60)

print("\nFinal container positions:")
for container in final_containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

final_violations = scheduler.count_priority_violations(final_containers)
print("\nSolution quality:")
print(f"  Initial priority violations: {initial_violations}")
print(f"  Final priority violations: {final_violations}")

if final_violations == 0:
    print("  All counted priority violations have been eliminated.")
else:
    print(f"  The final configuration has {final_violations} remaining violations.")

visualize_algorithm_progression(
    containers, final_containers, move_history, stacks, tiers
)

## Interpreting the method

The execution log reports the selected container, source and destination stacks, candidate score, and violation count before and after each move. A positive change in the violation term means the move reduced the counted conflicts; a move can still be selected even when it does not reduce them, because selection uses the full composite score. The final violation count is therefore an essential result to inspect, rather than assuming that the heuristic always reaches a violation-free arrangement.

The elapsed time is the runtime of this single execution in the current environment. It is not, by itself, evidence of performance on larger instances. The move history and state visualizations help explain the decisions made for this example, while the scoring rule makes the local trade-offs explicit.

### Strengths and limitations

**Strengths**

- The method evaluates accessible moves directly and has a transparent, inspectable scoring rule.
- Its move limit and feasibility checks make the execution bounded and keep moves within stack-capacity and accessibility constraints.
- The score can reflect several operational preferences through explicit weighted terms.

**Limitations**

- A greedy choice considers the current candidate scores, not the full consequences of a sequence of future moves; it provides no optimality guarantee.
- The heuristic may fail to eliminate all violations within the move limit, may revisit configurations, and may select a move that temporarily increases violations.
- Results depend on the scoring weights and on the simplified assumptions used for movement efficiency and stack balance.
- The move limit is a stopping safeguard, not a proof that a solution exists or that the best reachable arrangement has been found.

### When this method is useful

List scheduling is useful when a fast, understandable sequence of feasible moves is valuable and a heuristic result is acceptable. Its output can support operational planning or provide a baseline for examining a small instance. When interpreting any run, check whether the stopping condition was zero violations, lack of feasible moves, or the move limit; these outcomes have different meanings.